In [11]:
import os
from glob import glob
import nibabel as nib
import numpy as np


RAT = "R107"


ROOT = r"C:\Users\Estudiantes\Desktop\Estudio_tiemposdedifusion"
rat_path = os.path.join(ROOT, RAT)

print("\nProcesando:", RAT)

# =========================
# Buscar máscara
# =========================
mask_candidates = glob(os.path.join(
    rat_path,
    "sourcedata",
    "*",
    "preproc",
    "dwi",
    "*_dwi_mask.nii*"
))

if not mask_candidates:
    raise FileNotFoundError(f"No se encontró máscara para {RAT}")

mask_path = mask_candidates[0]
mask_folder = os.path.dirname(mask_path)

print("Máscara:", mask_path)

mask_nii = nib.load(mask_path)
mask = mask_nii.get_fdata()

# =========================
# DWIs preproc
# =========================
dwi_dir = os.path.join(rat_path, "derivatives", "dwi")
dwi_files = glob(os.path.join(dwi_dir, "*_dwi_preproc.nii*"))

for dwi_path in dwi_files:
    
    if "_masked" in dwi_path:
        continue
    
    print("\nProcesando:", os.path.basename(dwi_path))
    
    img_nii = nib.load(dwi_path)
    img = img_nii.get_fdata()
    
    if img.shape[:3] != mask.shape[:3]:
        print("⚠ Dimensiones incompatibles")
        continue
    
    if img.ndim == 4:
        masked = img * mask[..., np.newaxis]
    else:
        masked = img * mask
    
    # Guardar en la carpeta de la máscara
    base_name = os.path.basename(dwi_path).replace(".nii.gz", "")
    out_path = os.path.join(mask_folder, base_name + "_masked.nii.gz")
    
    out_nii = nib.Nifti1Image(
        masked.astype(np.float32),
        img_nii.affine,
        img_nii.header
    )
    
    nib.save(out_nii, out_path)
    
    print("✅ Guardado en sourcedata:", os.path.basename(out_path))

print("\n Terminado para", RAT)


Procesando: R107
Máscara: C:\Users\Estudiantes\Desktop\Estudio_tiemposdedifusion\R107\sourcedata\sub-G230924_R107_Tiemposdifusion_2_2_230924\preproc\dwi\sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-12_run-1_dwi_mask.nii.gz

Procesando: sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-12_run-1_dwi_preproc.nii.gz
✅ Guardado en sourcedata: sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-12_run-1_dwi_preproc_masked.nii.gz

Procesando: sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-15_run-1_dwi_preproc.nii.gz
✅ Guardado en sourcedata: sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-15_run-1_dwi_preproc_masked.nii.gz

Procesando: sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-18_run-1_dwi_preproc.nii.gz
✅ Guardado en sourcedata: sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-18_run-1_dwi_preproc_masked.nii.gz

Procesando: sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-8_run-1_dwi_preproc.nii.gz
✅ Guardado en sourcedata: sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-8_run-1_dwi_prepr